# VWAP Reclaim Notebook

In [ ]:
import os
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def find_repo_root(start: Path) -> Path:
    target = Path("scripts/trading_framework/config/sessions.yaml")
    for p in [start, *start.parents]:
        if (p / target).exists():
            return p
    raise FileNotFoundError("Could not locate repo root containing sessions.yaml")

ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
CONFIG_PATH = ROOT / "scripts" / "trading_framework" / "config" / "sessions.yaml"

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from scripts.trading_framework.config.config_loader import load_config
from scripts.libs_py.data.loader import DataLoader
from scripts.libs_py.features.feature_registry import FeatureRegistry
from scripts.strategies.vwap_reclaim.core.vwap_reclaim import VWAPReclaimStrategy
from scripts.trading_framework.core.mfe_mae import compute_mfe_mae_rich, summarize_mfe_mae_rich
from scripts.trading_framework.core.signal_adapter import enrich_signals, split_approved_vetoed

SYMBOL = "ES"
MAX_SIGNALS = 50
MAX_FORWARD_BARS = 120
FEATURES_NEEDED = [
    "atr_14",
    "vwap",
    "vwap_distance",
    "vwap_distance_atr",
    "vwap_cross_count",
    "above_vwap",
    "chop_score",
    "chop_regime",
    "chop_vwap_flag",
    "bb_upper",
    "bb_lower",
    "bb_pct_b",
    "ib_high",
    "ib_low",
    "ib_mid",
    "ib_width",
    "ib_formed",
]

print("Repo root:", ROOT)
print("CWD:", Path.cwd())
print("Config path:", CONFIG_PATH)
print("Notebook symbol:", SYMBOL)
print("Max signals per bucket:", MAX_SIGNALS)

In [ ]:
cfg = load_config(str(CONFIG_PATH))
point_value = cfg.execution.point_value.get(SYMBOL, 50.0)

loader = DataLoader(cfg)
df = loader.load_enriched(SYMBOL)
registry = FeatureRegistry(cfg)
df = registry.ensure_features(df, FEATURES_NEEDED)

strategy = VWAPReclaimStrategy(ticker=SYMBOL)
raw_signals = strategy.hunt(df)
enriched = enrich_signals(
    raw_signals,
    df,
    strategy_name="vwap_reclaim",
    symbol=SYMBOL,
    point_value=point_value,
)
approved, vetoed = split_approved_vetoed(enriched)

approved_run = approved.head(MAX_SIGNALS).copy()
vetoed_run = vetoed.head(MAX_SIGNALS).copy()

approved_results = compute_mfe_mae_rich(
    df,
    approved_run,
    max_forward_bars=MAX_FORWARD_BARS,
    horizons=cfg.mfe_mae.forward_horizons_minutes,
    atr_col="atr_14",
)
vetoed_results = compute_mfe_mae_rich(
    df,
    vetoed_run,
    max_forward_bars=MAX_FORWARD_BARS,
    horizons=cfg.mfe_mae.forward_horizons_minutes,
    atr_col="atr_14",
)
summary_approved = summarize_mfe_mae_rich(approved_results)
summary_vetoed = summarize_mfe_mae_rich(vetoed_results)


def summary_table(summary: dict) -> pd.DataFrame:
    if not summary:
        return pd.DataFrame()
    rows = [
        {
            "metric": "mfe_points",
            "p25": summary.get("mfe_p25"),
            "p50": summary.get("mfe_p50"),
            "p75": summary.get("mfe_p75"),
        },
        {
            "metric": "mae_points",
            "p25": summary.get("mae_p25"),
            "p50": summary.get("mae_p50"),
            "p75": summary.get("mae_p75"),
        },
        {
            "metric": "mfe_pct",
            "p25": summary.get("mfe_pct_p25"),
            "p50": summary.get("mfe_pct_p50"),
            "p75": summary.get("mfe_pct_p75"),
        },
        {
            "metric": "mae_pct",
            "p25": summary.get("mae_pct_p25"),
            "p50": summary.get("mae_pct_p50"),
            "p75": summary.get("mae_pct_p75"),
        },
        {
            "metric": "mfe_atr",
            "p25": summary.get("mfe_atr_p25"),
            "p50": summary.get("mfe_atr_p50"),
            "p75": summary.get("mfe_atr_p75"),
        },
        {
            "metric": "mae_atr",
            "p25": summary.get("mae_atr_p25"),
            "p50": summary.get("mae_atr_p50"),
            "p75": summary.get("mae_atr_p75"),
        },
    ]
    return pd.DataFrame(rows).round(4)


def conditional_tables(frame: pd.DataFrame, results: list) -> dict[str, pd.DataFrame]:
    if frame.empty or not results:
        return {}

    merged = frame.head(len(results)).copy()
    merged["peak_mfe"] = [r.mfe_points[-1] if r.mfe_points else 0.0 for r in results]
    merged["peak_mae"] = [r.mae_points[-1] if r.mae_points else 0.0 for r in results]
    merged["peak_mfe_pct"] = [r.mfe_pct[-1] if r.mfe_pct else 0.0 for r in results]
    merged["peak_mae_pct"] = [r.mae_pct[-1] if r.mae_pct else 0.0 for r in results]
    merged["reached_1r"] = [r.reached_1r for r in results]
    merged["reached_2r"] = [r.reached_2r for r in results]

    tables = {}
    for group_col in ["context_session_block", "context_chop_regime", "direction", "context_chop_score"]:
        if group_col not in merged.columns:
            continue
        grouped = merged.groupby(group_col, dropna=False).agg(
            count=("peak_mfe", "size"),
            mfe_median=("peak_mfe", "median"),
            mae_median=("peak_mae", "median"),
            mfe_pct_median=("peak_mfe_pct", "median"),
            mae_pct_median=("peak_mae_pct", "median"),
            pct_reach_1r=("reached_1r", "mean"),
            pct_reach_2r=("reached_2r", "mean"),
        ).round(4)
        if not grouped.empty:
            tables[group_col] = grouped
    return tables


veto_counts = vetoed["veto_reason"].value_counts().rename_axis("veto_reason").reset_index(name="count")
key_metrics = pd.DataFrame(
    [
        {
            "bucket": "approved",
            "raw_signal_count": len(approved),
            "analyzed_signals": len(approved_results),
            "pct_reach_1r": summary_approved.get("pct_reach_1r"),
            "pct_reach_2r": summary_approved.get("pct_reach_2r"),
            "pct_reach_3r": summary_approved.get("pct_reach_3r"),
            "avg_time_to_1r": summary_approved.get("avg_time_to_1r"),
            "optimal_stop_atr": summary_approved.get("optimal_stop_atr"),
            "optimal_stop_pct": summary_approved.get("optimal_stop_pct"),
        },
        {
            "bucket": "vetoed",
            "raw_signal_count": len(vetoed),
            "analyzed_signals": len(vetoed_results),
            "pct_reach_1r": summary_vetoed.get("pct_reach_1r"),
            "pct_reach_2r": summary_vetoed.get("pct_reach_2r"),
            "pct_reach_3r": summary_vetoed.get("pct_reach_3r"),
            "avg_time_to_1r": summary_vetoed.get("avg_time_to_1r"),
            "optimal_stop_atr": summary_vetoed.get("optimal_stop_atr"),
            "optimal_stop_pct": summary_vetoed.get("optimal_stop_pct"),
        },
    ]
).round(4)

app_ratio = summary_approved.get("mfe_p50", 0.0) / max(summary_approved.get("mae_p50", 0.001), 0.001)
vet_ratio = summary_vetoed.get("mfe_p50", 0.0) / max(summary_vetoed.get("mae_p50", 0.001), 0.001)
filter_effectiveness = pd.DataFrame(
    [
        {"bucket": "approved", "median_mfe_mae_ratio": round(app_ratio, 4)},
        {"bucket": "vetoed", "median_mfe_mae_ratio": round(vet_ratio, 4)},
    ]
)

approved_tables = conditional_tables(approved_run, approved_results)

sample_result = pd.DataFrame()
if approved_results:
    sample = approved_results[0]
    sample_result = pd.DataFrame(
        [
            {
                "signal_time": sample.signal_time,
                "direction": sample.direction,
                "entry_price": sample.entry_price,
                "stop_price": sample.stop_price,
                "risk_points": sample.risk_points,
                "mfe_peak_points": sample.mfe_points[-1] if sample.mfe_points else 0.0,
                "mae_peak_points": sample.mae_points[-1] if sample.mae_points else 0.0,
                "mfe_peak_pct": sample.mfe_pct[-1] if sample.mfe_pct else 0.0,
                "mae_peak_pct": sample.mae_pct[-1] if sample.mae_pct else 0.0,
                "reached_1r": sample.reached_1r,
                "reached_2r": sample.reached_2r,
                "reached_3r": sample.reached_3r,
                "time_to_1r": sample.time_to_1r,
                "time_to_2r": sample.time_to_2r,
                "time_to_3r": sample.time_to_3r,
                "mfe_peak_bar": sample.mfe_peak_bar,
                "mae_trough_bar": sample.mae_trough_bar,
                "path_len": len(sample.path),
            }
        ]
    )

output_dir = ROOT / "reports" / "vwap_reclaim" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

bundle_path = output_dir / "notebook_results_preview.pkl"
pd.to_pickle(
    {
        "approved_results": approved_results,
        "vetoed_results": vetoed_results,
        "summary_approved": summary_approved,
        "summary_vetoed": summary_vetoed,
        "approved_signals": approved_run,
        "vetoed_signals": vetoed_run,
    },
    bundle_path,
)

display(Markdown(f"## Raw Analysis Results for {SYMBOL}"))
display(key_metrics)

display(Markdown("### Signal Counts"))
display(pd.DataFrame([{"raw_signals": len(raw_signals), "approved": len(approved), "vetoed": len(vetoed)}]))

display(Markdown("### Top Veto Reasons"))
display(veto_counts.head(10))

display(Markdown("### Approved Summary"))
display(summary_table(summary_approved))

display(Markdown("### Vetoed Summary"))
display(summary_table(summary_vetoed))

display(Markdown("### Filter Effectiveness"))
display(filter_effectiveness)

if approved_results and summary_approved.get("winner_2r_mae_p50") is not None:
    winner_heat = pd.DataFrame(
        [
            {
                "winner_2r_mae_p50": summary_approved.get("winner_2r_mae_p50"),
                "winner_2r_mae_p75": summary_approved.get("winner_2r_mae_p75"),
                "winner_2r_mae_p90": summary_approved.get("winner_2r_mae_p90"),
                "winner_2r_mae_pct_p50": summary_approved.get("winner_2r_mae_pct_p50"),
                "winner_2r_mae_pct_p75": summary_approved.get("winner_2r_mae_pct_p75"),
                "winner_2r_mae_pct_p90": summary_approved.get("winner_2r_mae_pct_p90"),
            }
        ]
    ).round(4)
    display(Markdown("### Winner Heat Analysis"))
    display(winner_heat)

if not sample_result.empty:
    display(Markdown("### Sample Rich MFE/MAE Object"))
    display(sample_result.round(4))

for group_col, table in approved_tables.items():
    display(Markdown(f"### Conditional Table: {group_col}"))
    display(table)

print(f"Results saved to: {bundle_path}")